# Yet New — fine-tune the map-editor assistant (Google Colab or Kaggle, free T4 GPU)

Teaches a small model (**Qwen2.5-3B-Instruct**, configurable) to turn what venue managers type — casual, short, with typos — into the editor's actions, the same contract as the Claude route (`src/lib/assistant/assistantSpec.ts`). The result runs for free in Ollama next to the layer labeler.

**Data:** `python scripts/make_assistant_data.py` in `ml/`, then zip `data/assistant` (`train.jsonl`, `test.jsonl`, `manifest.json`). It is *generated* data: every answer is correct by construction, and no customer data is involved. The test file holds out whole phrasings, so the score below is on wordings the model never saw.

**Kaggle** (30 free GPU hours a week, runs up to 12 h with the browser closed): upload `assistant_dataset.zip` as a dataset (**+ Add Input → Upload**), Settings → Accelerator **GPU T4 x2**, Internet **on**, then **Save Version → Save & Run All (Commit)**. The files appear under the version's **Output**.

**Colab:** Runtime → T4 GPU → Run all, and pick the zip when asked. Set `SAVE_TO_DRIVE = True` to keep progress in Google Drive: if the session is cut off, Run all again continues from the last saved step.

**Smoke mode** (`SMOKE_TEST = True`): a few training steps to prove the environment works; nothing is exported, the output is named `SMOKE-TEST-delete-me-assistant`, and the service refuses any model with "smoke" in its name.

**Pinned 2026-10-10** (same as the layer notebook; read from the wheels, not run here — no GPU): `unsloth 2026.9.11` → `trl ≤ 0.24.0`, `transformers ≤ 5.5.0`, `datasets < 4.4.0`, `torch < 2.13`. Measured on a Colab T4: ~108 s per training step (157 steps per epoch). **Not verified:** resuming from a checkpoint, the Kaggle run, and the GGUF file name Unsloth writes.

The prompts are **not** built here: each row in `train.jsonl` was made by `assistant.training_example()` — the same code the service uses at inference.

In [ ]:
!pip install -q "unsloth==2026.9.11" "unsloth_zoo==2026.9.7" "trl==0.24.0" "transformers==5.5.0" "peft==0.21.0" \
    "datasets==4.3.0" "bitsandbytes==0.50.2" "accelerate==1.15.0" "gguf==0.19.0"

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle "T4 x2": train on one GPU

import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU: Colab → Runtime → Change runtime type → T4 GPU; Kaggle → Settings → Accelerator → GPU T4 x2"

## Settings

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"   # the same family the service runs (qwen2.5:3b)
EPOCHS = 2              # 2–3
LEARNING_RATE = 2e-4
LORA_R = 16
MAX_SEQ_LENGTH = 4096   # instructions ≈ 1,900 tokens + the map summary + the answer
BATCH_SIZE = 2
GRAD_ACCUM = 8
SEED = 3407
VAL_SAMPLE = 60         # test requests to score here (generation is slow)

SAVE_TO_DRIVE = False   # Colab: keep checkpoints in Google Drive, so a cut-off session continues where it stopped
SAVE_STEPS = 20         # a checkpoint every N steps (~35 min on a T4)

SMOKE_TEST = False
SMOKE_STEPS = 5

OUTPUT_NAME = "SMOKE-TEST-delete-me-assistant" if SMOKE_TEST else "yetnew-assistant"
print("SMOKE TEST — environment check only; nothing will be exported." if SMOKE_TEST else f"Real training run → {OUTPUT_NAME}")

## 1. Upload the dataset

In [ ]:
import hashlib, json, pathlib, zipfile

ON_KAGGLE = pathlib.Path("/kaggle/input").exists()
if ON_KAGGLE:
    # + Add Input → the dataset made from assistant_dataset.zip (Kaggle usually unzips it; if not, it is unzipped here).
    for z in sorted(pathlib.Path("/kaggle/input").rglob("*.zip")):
        zipfile.ZipFile(z).extractall("dataset")
    search = [pathlib.Path("/kaggle/input"), pathlib.Path("dataset")]
else:
    from google.colab import files
    print("Upload the zipped data/assistant folder (train.jsonl, test.jsonl, manifest.json).")
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("dataset")
    search = [pathlib.Path("dataset")]
found = [p.parent for d in search if d.exists() for p in d.rglob("manifest.json")
         if "spec_version" in json.loads(p.read_text(encoding="utf-8"))]
assert found, "No assistant dataset found (Kaggle: + Add Input → your assistant_dataset.zip dataset)"
root = found[0]
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
assert "spec_version" in manifest and "held_out_templates" in manifest, "Not an assistant dataset (make_assistant_data.py)"

def load(name):
    return [json.loads(line) for line in (root / name).read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows, test_rows = load("train.jsonl"), load("test.jsonl")
held = set(manifest["held_out_templates"])
assert not {r["template"] for r in train_rows} & held, "A held-out phrasing is in the training data"
DATA_ID = hashlib.sha256((root / "train.jsonl").read_bytes()).hexdigest()[:10]
print(f"spec version {manifest['spec_version']} · {len(train_rows)} training / {len(test_rows)} test requests · {len(held)} phrasings held out · data {DATA_ID}")

## 2. Load the base model and check the chat format

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None)

# Training text must be exactly what Ollama sends at inference (ml/yetnew_layers/chat_format.expected).
def expected(messages, add_generation_prompt):
    text = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return text + ("<|im_start|>assistant\n" if add_generation_prompt else "")

def as_text(messages, add_generation_prompt=False):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)

sample = train_rows[0]["messages"]
assert as_text(sample) == expected(sample, False), "The tokenizer's chat template is not the expected Qwen2.5 format"
assert as_text(sample[:2], True) == expected(sample[:2], True)
lengths = [len(tokenizer(as_text(r["messages"]), add_special_tokens=False)["input_ids"]) for r in train_rows[:300] + test_rows[:100]]
print("longest example (sampled):", max(lengths), "tokens")
assert max(lengths) <= MAX_SEQ_LENGTH, "Raise MAX_SEQ_LENGTH"

## 3. LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)

## 4. Train (loss on the answer only)

In [ ]:
import glob
from datasets import Dataset

# Checkpoints belong to one run: same data, epochs, rate and LoRA size — anything else starts over.
RUN_ID = f"{OUTPUT_NAME}-{DATA_ID}-e{EPOCHS}-lr{LEARNING_RATE}-r{LORA_R}-s{SEED}"
if SAVE_TO_DRIVE and not ON_KAGGLE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINTS = f"/content/drive/MyDrive/yetnew-checkpoints/{RUN_ID}"
else:
    CHECKPOINTS = f"checkpoints/{RUN_ID}"
saved = sorted(glob.glob(f"{CHECKPOINTS}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = saved[-1] if saved and not SMOKE_TEST else None
print(f"continuing from {resume}" if resume else f"starting fresh · checkpoints in {CHECKPOINTS}")
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

train_ds = Dataset.from_list([{"text": as_text(r["messages"]).removesuffix("\n")} for r in train_rows])
trainer = SFTTrainer(
    model=model, processing_class=tokenizer, train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LENGTH, packing=False,
        num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE, lr_scheduler_type="linear", warmup_steps=0.05, weight_decay=0.01,
        optim="adamw_8bit", logging_steps=5, seed=SEED, output_dir=CHECKPOINTS, save_strategy="no" if SMOKE_TEST else "steps", save_steps=SAVE_STEPS, save_total_limit=1, report_to="none",
        max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

ex = trainer.train_dataset[0]
kept = tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]).strip()
print("trained on:", kept[:200])
assert kept.startswith(train_rows[0]["messages"][2]["content"]), "Loss must be on the JSON answer only"

stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

## 5. Held-out phrasings

Indicative: the decision is made by `scripts/eval_assistant.py` through Ollama on your machine (the same requests, more of them).

In [ ]:
from collections import defaultdict

FastLanguageModel.for_inference(model)

def answer(messages):
    ids = tokenizer(as_text(messages[:2], add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=400, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def norm(actions):
    return [{k: (v.strip().lower() if isinstance(v, str) else v) for k, v in a.items()} for a in actions]

if SMOKE_TEST:
    print(f"SMOKE TEST: {SMOKE_STEPS} steps — these numbers mean nothing.")
per_op, wrong, valid = defaultdict(lambda: [0, 0]), [], 0
sample = test_rows[:3] if SMOKE_TEST else test_rows[:VAL_SAMPLE]
for r in sample:
    try:
        got = json.loads(answer(r["messages"])); valid += 1
    except json.JSONDecodeError:
        got = {"actions": [], "reply": None}
    want = json.loads(r["messages"][2]["content"])
    ok = (not want["actions"] and not got.get("actions") and bool(got.get("reply"))) if want["reply"] else norm(got.get("actions") or []) == norm(want["actions"])
    per_op[r["op"]][1] += 1
    per_op[r["op"]][0] += ok
    if not ok:
        wrong.append((r["body"]["text"], want, got))

right = sum(c for c, _ in per_op.values())
print(f"exact {right}/{len(sample)} = {right / len(sample):.1%} · valid JSON {valid}/{len(sample)}")
print("  " + "  ".join(f"{op} {c}/{n}" for op, (c, n) in sorted(per_op.items())))
for text, want, got in wrong[:20]:
    print(f"  {text!r}\n     want {want}\n     got  {got}")

## 6. Export: LoRA adapter + GGUF q4_k_m

In [ ]:
import glob, shutil
from gguf import GGUFReader
from jinja2.sandbox import ImmutableSandboxedEnvironment

if SMOKE_TEST:
    raise SystemExit(
        "SMOKE TEST finished: the environment works. Nothing was exported, on purpose.\n"
        "DELETE IT NOW: Colab → Runtime → Disconnect and delete runtime (Kaggle: delete this version) (removes the SMOKE-TEST-delete-me-assistant weights).\n"
        "Then set SMOKE_TEST = False and train for real."
    )
assert "smoke" not in OUTPUT_NAME.lower()

model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
shutil.make_archive(f"{OUTPUT_NAME}-lora", "zip", "lora_adapter")
# The merged model and the f16 GGUF are ~12 GB: on Kaggle they go to /tmp, not the 20 GB output folder.
GGUF_DIR = "/tmp/gguf_out" if ON_KAGGLE else "gguf_out"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
src = sorted(p for d in (GGUF_DIR, ".", "/tmp") for p in glob.glob(f"{d}/**/*.gguf", recursive=True) if "q4_k_m" in p.lower())[0]

embedded = GGUFReader(src).fields["tokenizer.chat_template"].contents()
render = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True).from_string(embedded).render
for r in test_rows[:5]:
    assert render(messages=r["messages"][:2], add_generation_prompt=True, tools=None) == expected(r["messages"][:2], True)
print("Embedded chat template renders the training prompt ✓")

final = f"{OUTPUT_NAME}.Q4_K_M.gguf"
shutil.copy(src, final)
if ON_KAGGLE:
    shutil.rmtree("checkpoints", ignore_errors=True); shutil.rmtree("lora_adapter", ignore_errors=True)
    print(f"Done. Download {final} and {OUTPUT_NAME}-lora.zip from this version's Output tab.")
else:
    files.download(f"{OUTPUT_NAME}-lora.zip")
    files.download(final)

## 7. On your machine

```powershell
# put yetnew-assistant.Q4_K_M.gguf in ml/ollama/, then from ml/:
ollama create yetnew-assistant -f ollama/Modelfile.assistant
.venv\Scripts\python.exe scripts\check_template.py --model yetnew-assistant     # REQUIRED after every ollama create / Ollama upgrade
.venv\Scripts\python.exe scripts\eval_assistant.py --finetuned yetnew-assistant --write
```

The service switches to the fine-tuned assistant only if `eval_assistant.py` wrote an `assistant_choice.json` in which it won.